# Songo Lot 39 — GPU parallel MCTS scaling
Exécuter les cellules dans l'ordre. Chaque niveau est persistant et reprenable sur Drive.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL='https://github.com/GlennEriss/songo-fish.git'
REPO_DIR=Path('/content/songo-fish')
if not REPO_DIR.exists(): subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)],check=True)
else: subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[train,perf,dev]'],check=True)
os.environ['PYTHONPATH']='packages:apps/trainer/scripts'


In [ ]:
from google.colab import drive, files
DRIVE_AVAILABLE=Path('/content/drive/MyDrive').is_dir()
if not DRIVE_AVAILABLE:
    try:
        drive.mount('/content/drive',timeout_ms=300000)
        DRIVE_AVAILABLE=Path('/content/drive/MyDrive').is_dir()
    except Exception as exc: print('Drive indisponible, fallback upload:',exc)
if DRIVE_AVAILABLE:
    ROOT=Path('/content/drive/MyDrive/songo-ai'); INPUT=ROOT/'inputs/lot39_inputs.tar.gz'; OUTPUT=ROOT/'experiments/lot39_gpu_parallel_scaling'; EXPORT=ROOT/'exports/lot39_results.tar.gz'
else:
    files.upload(); INPUT=Path('/content/lot39_inputs.tar.gz'); OUTPUT=Path('/content/lot39_gpu_parallel_scaling'); EXPORT=Path('/content/lot39_results.tar.gz')
subprocess.run(['tar','-xzf',str(INPUT),'-C',str(REPO_DIR)],check=True)
OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
import time
RUN=[sys.executable,'-u','apps/trainer/scripts/run_srn_lot39.py']
def supervised_run(arguments, label):
    log=Path('/content')/f'{label}.log'
    with log.open('wb') as stream:
        process=subprocess.Popen(arguments,stdout=stream,stderr=subprocess.STDOUT,cwd=REPO_DIR)
        started=time.monotonic(); offset=0
        while process.poll() is None:
            time.sleep(10)
            data=log.read_bytes()[offset:]
            if data:
                print(data.decode(errors='replace'),end='',flush=True); offset+=len(data)
            print(f'[{label}] actif depuis {time.monotonic()-started:.0f}s, pid={process.pid}',flush=True)
        data=log.read_bytes()[offset:]
        if data: print(data.decode(errors='replace'),end='',flush=True)
    print(f'[{label}] code de sortie={process.returncode}; journal={log}',flush=True)
    if process.returncode: raise RuntimeError(f'{label} a échoué avec le code {process.returncode}; consulter {log}')
supervised_run(RUN+['--stage','prepare','--device','cuda','--positions-file','data/colab_bridge/lot39_benchmark_positions.json','--output',str(OUTPUT)],'lot39_prepare')
supervised_run(RUN+['--stage','correctness','--device','cuda','--output',str(OUTPUT)],'lot39_correctness')


## Phase A — stages indépendants
Chaque cellule ignore automatiquement un résultat COMPLETE valide.

In [ ]:
supervised_run(RUN+['--stage','run','--device','cuda','--concurrency','16','--output',str(OUTPUT)],'lot39_n16')

In [ ]:
supervised_run(RUN+['--stage','run','--device','cuda','--concurrency','32','--output',str(OUTPUT)],'lot39_n32')

In [ ]:
supervised_run(RUN+['--stage','run','--device','cuda','--concurrency','64','--output',str(OUTPUT)],'lot39_n64')

In [ ]:
supervised_run(RUN+['--stage','run','--device','cuda','--concurrency','128','--output',str(OUTPUT)],'lot39_n128')

In [ ]:
supervised_run(RUN+['--stage','run','--device','cuda','--concurrency','256','--output',str(OUTPUT)],'lot39_n256')

In [ ]:
subprocess.run(RUN+['--stage','finalize','--output',str(OUTPUT)],check=True)
subprocess.run(RUN+['--stage','export','--output',str(OUTPUT),'--bundle',str(EXPORT)],check=True)
if not DRIVE_AVAILABLE:
    files.download(str(EXPORT)); files.download(str(Path(str(EXPORT)+'.sha256')))
